In [1]:
# ==========================================
# PHASE 5 — COMPOSITE RISK + SURVIVAL
# ==========================================

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

# Load clean dataset
df = pd.read_csv("../data/5-star edible sushi.csv")
df["report_month"] = pd.to_datetime(df["report_month"])

# Load trained Phase 4 models
checkpoint = joblib.load("../models/phase4_checkpoint.joblib")

cost_final_model = checkpoint["cost_final_model"]
schedule_final_model = checkpoint["schedule_final_model"]

print("✅ Phase 5 objects loaded")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Cost model:", type(cost_final_model))
print("Schedule model:", type(schedule_final_model))

✅ Phase 5 objects loaded
Rows: 109787
Columns: 52
Cost model: <class 'sklearn.pipeline.Pipeline'>
Schedule model: <class 'sklearn.pipeline.Pipeline'>


In [2]:
# ==========================================
# RECREATE MODEL FEATURES
# ==========================================

drop_cols = [
    "project_id",
    "project_name",
    "report_month",
    "target_cost_overrun_12m",
    "target_schedule_risk_12m",
    "cost_target_valid",
    "serial_no",
    "source_file",
    "source_page",
    "implementing_agency"
]

drop_cols = [
    c for c in drop_cols
    if c in df.columns
]

X = df.drop(columns=drop_cols)

print("Model feature matrix:", X.shape)
print("\nFeatures:")
print(list(X.columns))

Model feature matrix: (109787, 47)

Features:
['month', 'year', 'sector', 'state', 'approval_year', 'project_age', 'original_duration_months', 'duration_overrun_months', 'original_cost_crore', 'revised_cost_crore', 'anticipated_cost_crore', 'current_cost', 'cumulative_expenditure_crore', 'anticipated_delay_from_original', 'remaining_org_months', 'delay_revised_months', 'delay_revisied_months', 'milestones_achieved', 'milestones_total', 'milestone_completion_percentage', 'milestone_data_reliable', 'cost_revision_percentage', 'anticipated_cost_percentage', 'exp_vs_anti_prct', 'exp_vs_org_prct', 'exp_vs_rev_prct', 'project_duration_elapsed_percentage', 'cost_growth_3m', 'cost_growth_6m', 'cost_growth_12m', 'expenditure_growth_3m', 'expenditure_growth_6m', 'expenditure_growth_12m', 'schedule_change_3m', 'schedule_change_6m', 'schedule_change_12m', 'milestone_progress_change_3m', 'milestone_progress_change_6m', 'milestone_progress_change_12m', 'expenditure_progress_gap_org', 'expenditure_pr

In [3]:
# ==========================================
# GENERATE RISK PROBABILITIES
# ==========================================

# Cost probability
cost_probability = cost_final_model.predict_proba(X)[:, 1]

# Schedule probability
schedule_probability = schedule_final_model.predict_proba(X)[:, 1]

df["cost_risk_probability"] = cost_probability
df["schedule_risk_probability"] = schedule_probability

print("✅ Probabilities generated")

print("\nCost probability:")
print(df["cost_risk_probability"].describe())

print("\nSchedule probability:")
print(df["schedule_risk_probability"].describe())

✅ Probabilities generated

Cost probability:
count    109787.000000
mean          0.109220
std           0.239288
min           0.000004
25%           0.003903
50%           0.015359
75%           0.066267
max           0.999617
Name: cost_risk_probability, dtype: float64

Schedule probability:
count    109787.000000
mean          0.308309
std           0.351013
min           0.000013
25%           0.013148
50%           0.128203
75%           0.595605
max           0.998225
Name: schedule_risk_probability, dtype: float64


In [4]:
# ==========================================
# 5.1 COMPOSITE RISK SCORE
# ==========================================

COST_WEIGHT = 0.50
SCHEDULE_WEIGHT = 0.50

df["composite_risk_score"] = (
    COST_WEIGHT * df["cost_risk_probability"] +
    SCHEDULE_WEIGHT * df["schedule_risk_probability"]
) * 100

print("Composite score created.")

print("\nComposite Risk Score:")
print(
    df["composite_risk_score"].describe()
)

print("\nWeighting logic:")
print("Cost Overrun Risk :", COST_WEIGHT)
print("Schedule Risk     :", SCHEDULE_WEIGHT)
print("Total             :", COST_WEIGHT + SCHEDULE_WEIGHT)

Composite score created.

Composite Risk Score:
count    109787.000000
mean         20.876438
std          22.606377
min           0.004162
25%           1.919958
50%          10.804028
75%          40.319302
max          99.632652
Name: composite_risk_score, dtype: float64

Weighting logic:
Cost Overrun Risk : 0.5
Schedule Risk     : 0.5
Total             : 1.0


In [6]:
# ==========================================
# RECREATE PHASE 2 HELD-OUT TEST SETS
# ==========================================

split_date = pd.Timestamp("2021-01-01")

# Cost target
cost_mask = df["target_cost_overrun_12m"].notna()

cost_test_mask = (
    cost_mask &
    (df["report_month"] >= split_date)
)

X_test = X.loc[cost_test_mask].copy()

y_test = df.loc[
    cost_test_mask,
    "target_cost_overrun_12m"
].astype(int)

# Schedule target
schedule_mask = df["target_schedule_risk_12m"].notna()

schedule_test_mask = (
    schedule_mask &
    (df["report_month"] >= split_date)
)

X_schedule_test = X.loc[schedule_test_mask].copy()

y_schedule_test = df.loc[
    schedule_test_mask,
    "target_schedule_risk_12m"
].astype(int)

print("Cost test:", X_test.shape)
print("Schedule test:", X_schedule_test.shape)

Cost test: (7393, 47)
Schedule test: (4825, 47)


In [7]:
# ==========================================
# 5.2 — HELD-OUT COMPOSITE SCORES
# ==========================================

cost_test_prob = cost_final_model.predict_proba(X_test)[:, 1]

schedule_test_prob = (
    schedule_final_model.predict_proba(X_schedule_test)[:, 1]
)

cost_test_scores = pd.DataFrame({
    "index": X_test.index,
    "cost_probability": cost_test_prob
}).set_index("index")

schedule_test_scores = pd.DataFrame({
    "index": X_schedule_test.index,
    "schedule_probability": schedule_test_prob
}).set_index("index")

test_scores = cost_test_scores.join(
    schedule_test_scores,
    how="inner"
)

test_scores["composite_score"] = (
    0.5 * test_scores["cost_probability"] +
    0.5 * test_scores["schedule_probability"]
) * 100

print("Held-out observations:", len(test_scores))

print("\nComposite score distribution:")
print(
    test_scores["composite_score"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    ).round(2)
)

Held-out observations: 4825

Composite score distribution:
count    4825.00
mean       20.46
std        21.05
min         0.03
25%         1.43
50%        11.77
75%        38.59
90%        48.73
95%        55.62
max        95.76
Name: composite_score, dtype: float64


In [8]:
# ==========================================
# EMPIRICAL RISK BY COMPOSITE SCORE
# ==========================================

test_analysis = test_scores.copy()

test_analysis["cost_actual"] = y_test.loc[
    test_analysis.index
].astype(int)

test_analysis["schedule_actual"] = y_schedule_test.loc[
    test_analysis.index
].astype(int)

test_analysis["score_band"] = pd.cut(
    test_analysis["composite_score"],
    bins=[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
    include_lowest=True
)

risk_band_summary = (
    test_analysis
    .groupby("score_band", observed=True)
    .agg(
        observations=("composite_score", "size"),
        avg_score=("composite_score", "mean"),
        cost_event_rate=("cost_actual", "mean"),
        schedule_event_rate=("schedule_actual", "mean")
    )
    .reset_index()
)

print("EMPIRICAL RISK BY COMPOSITE SCORE")
display(risk_band_summary.round(4))

EMPIRICAL RISK BY COMPOSITE SCORE


,score_band,observations,avg_score,cost_event_rate,schedule_event_rate
0,"(-0.001, 10.0]",2308,2.487700,0.0412,0.0680
1,"(10.0, 20.0]",497,14.873800,0.0624,0.2978
2,"(20.0, 30.0]",407,25.181000,0.0860,0.4300
3,"(30.0, 40.0]",480,35.366199,0.0979,0.5625
4,"(40.0, 50.0]",768,45.591202,0.0820,0.7721
5,"(50.0, 60.0]",174,54.091702,0.1264,0.7126
6,"(60.0, 70.0]",92,64.712402,0.4457,0.7283
7,"(70.0, 80.0]",47,74.905701,0.5106,0.8085
8,"(80.0, 90.0]",41,83.852097,0.6585,0.8293
9,"(90.0, 100.0]",11,93.147697,1.0000,1.0000


In [9]:
# ==========================================
# CANDIDATE RISK TIERS
# ==========================================

candidate_tiers = pd.DataFrame({
    "Tier": ["LOW", "WATCH", "ELEVATED", "CRITICAL"],
    "Lower": [0, 25, 50, 75],
    "Upper": [25, 50, 75, 100]
})

tier_results = []

for _, tier in candidate_tiers.iterrows():

    mask = (
        (test_analysis["composite_score"] >= tier["Lower"]) &
        (test_analysis["composite_score"] < tier["Upper"])
    )

    subset = test_analysis.loc[mask]

    tier_results.append({
        "Tier": tier["Tier"],
        "Score Range": f"{tier['Lower']}-{tier['Upper']}",
        "Observations": len(subset),
        "Avg Score": subset["composite_score"].mean(),
        "Cost Event Rate": subset["cost_actual"].mean(),
        "Schedule Event Rate": subset["schedule_actual"].mean()
    })

tier_results = pd.DataFrame(tier_results)

print("CANDIDATE RISK TIER PERFORMANCE")
display(tier_results.round(4))

CANDIDATE RISK TIER PERFORMANCE


,Tier,Score Range,Observations,Avg Score,Cost Event Rate,Schedule Event Rate
0,LOW,0-25,3005,5.871100,0.0456,0.1311
1,WATCH,25-50,1455,39.676800,0.0921,0.6522
2,ELEVATED,50-75,290,58.972198,0.2586,0.7207
3,CRITICAL,75-100,75,83.289001,0.6667,0.8667


In [10]:
# ==========================================
# FINAL COMPOSITE RISK TIERS
# ==========================================

def assign_risk_tier(score):

    if score < 25:
        return "LOW"
    elif score < 50:
        return "WATCH"
    elif score < 75:
        return "ELEVATED"
    else:
        return "CRITICAL"


df["risk_tier"] = df["composite_risk_score"].apply(
    assign_risk_tier
)

print("FINAL RISK TIER DISTRIBUTION")
display(
    df["risk_tier"]
    .value_counts()
    .reindex(["LOW", "WATCH", "ELEVATED", "CRITICAL"])
    .rename_axis("Risk Tier")
    .reset_index(name="Observations")
)

FINAL RISK TIER DISTRIBUTION


,Risk Tier,Observations
0,LOW,71188
1,WATCH,28195
2,ELEVATED,7476
3,CRITICAL,2928


In [12]:
# ==========================================
# PROJECT-LEVEL CURRENT RISK
# ==========================================

latest_project_risk = (
    df.sort_values("report_month")
      .groupby("project_id", as_index=False)
      .tail(1)
      .copy()
)

project_risk_output = latest_project_risk[
    [
        "project_id",
        "report_month",
        "cost_risk_probability",
        "schedule_risk_probability",
        "composite_risk_score",
        "risk_tier"
    ]
].copy()

project_risk_output = project_risk_output.sort_values(
    "composite_risk_score",
    ascending=False
)

print("PROJECT-LEVEL RISK OUTPUT")
print("Projects:", len(project_risk_output))

display(project_risk_output.head(20))

PROJECT-LEVEL RISK OUTPUT
Projects: 2836


,project_id,report_month,cost_risk_probability,schedule_risk_probability,composite_risk_score,risk_tier
38145,N22000156,2018-11-01,0.971205,0.975483,97.334381,CRITICAL
26266,N18000113,2016-06-01,0.975299,0.944567,95.993301,CRITICAL
26593,N18000129,2016-10-01,0.986527,0.927734,95.713066,CRITICAL
26150,N18000104,2016-06-01,0.877998,0.964887,92.144257,CRITICAL
40990,N22000198,2018-11-01,0.917295,0.902315,90.980545,CRITICAL
31573,N18000265,2022-05-01,0.889523,0.902699,89.611092,CRITICAL
16604,N16000136,2018-01-01,0.885265,0.886545,88.590477,CRITICAL
50879,N22000390,2022-01-01,0.887119,0.873905,88.051201,CRITICAL
54275,N22000469,2022-05-01,0.990984,0.717935,85.445930,CRITICAL
27472,N18000161,2016-10-01,0.723170,0.984683,85.392616,CRITICAL


In [13]:
# ==========================================
# PHASE 5.2 — FINAL TIER VALIDATION
# ==========================================

tier_validation = (
    test_analysis.assign(
        risk_tier=test_analysis["composite_score"].apply(assign_risk_tier)
    )
    .groupby("risk_tier")
    .agg(
        observations=("composite_score", "size"),
        avg_score=("composite_score", "mean"),
        cost_event_rate=("cost_actual", "mean"),
        schedule_event_rate=("schedule_actual", "mean")
    )
    .reindex(["LOW", "WATCH", "ELEVATED", "CRITICAL"])
    .reset_index()
)

print("=" * 65)
print("PHASE 5.2 — EARLY WARNING TIER VALIDATION")
print("=" * 65)

display(tier_validation.round(4))

PHASE 5.2 — EARLY WARNING TIER VALIDATION


,risk_tier,observations,avg_score,cost_event_rate,schedule_event_rate
0,LOW,3005,5.871100,0.0456,0.1311
1,WATCH,1455,39.676800,0.0921,0.6522
2,ELEVATED,290,58.972198,0.2586,0.7207
3,CRITICAL,75,83.289001,0.6667,0.8667


In [14]:
# ==========================================
# PHASE 5.2 — EXIT CHECK
# ==========================================

print("=" * 65)
print("PHASE 5.2 EXIT CHECK")
print("=" * 65)

print("\nRisk tiers:")
print("LOW       : 0–25")
print("WATCH     : 25–50")
print("ELEVATED  : 50–75")
print("CRITICAL  : 75–100")

print("\nHeld-out validation:")
display(tier_validation.round(4))

print("\n✅ Composite score: COMPLETE")
print("✅ Risk tiers: COMPLETE")
print("✅ Empirical validation: COMPLETE")

PHASE 5.2 EXIT CHECK

Risk tiers:
LOW       : 0–25
WATCH     : 25–50
ELEVATED  : 50–75
CRITICAL  : 75–100

Held-out validation:


,risk_tier,observations,avg_score,cost_event_rate,schedule_event_rate
0,LOW,3005,5.871100,0.0456,0.1311
1,WATCH,1455,39.676800,0.0921,0.6522
2,ELEVATED,290,58.972198,0.2586,0.7207
3,CRITICAL,75,83.289001,0.6667,0.8667



✅ Composite score: COMPLETE
✅ Risk tiers: COMPLETE
✅ Empirical validation: COMPLETE


In [15]:
# ==========================================
# PHASE 5.3 — COX SURVIVAL DATASET
# ==========================================

survival_df = df[
    [
        "project_id",
        "report_month",
        "anticipated_delay_from_original"
    ]
].copy()

survival_df = survival_df.sort_values(
    ["project_id", "report_month"]
)

# Delay event
survival_df["delay_event"] = (
    survival_df["anticipated_delay_from_original"] > 0
).astype(int)

print("Panel rows:", len(survival_df))
print("Projects:", survival_df["project_id"].nunique())

Panel rows: 109787
Projects: 2836


In [16]:
# ==========================================
# FIRST OBSERVATION + FIRST DELAY
# ==========================================

first_observation = (
    survival_df
    .groupby("project_id")["report_month"]
    .min()
    .rename("start_date")
)

first_delay = (
    survival_df[survival_df["delay_event"] == 1]
    .groupby("project_id")["report_month"]
    .min()
    .rename("event_date")
)

last_observation = (
    survival_df
    .groupby("project_id")["report_month"]
    .max()
    .rename("last_date")
)

cox_df = pd.concat(
    [first_observation, first_delay, last_observation],
    axis=1
).reset_index()

# Event occurred if a delay was observed
cox_df["event"] = cox_df["event_date"].notna().astype(int)

# Event date if event occurred, otherwise censoring date
cox_df["end_date"] = cox_df["event_date"].combine_first(
    cox_df["last_date"]
)

# Duration in months
cox_df["duration_months"] = (
    (cox_df["end_date"] - cox_df["start_date"])
    .dt.days / 30.44
)

print("Cox dataset:", cox_df.shape)

print("\nEvent distribution:")
print(cox_df["event"].value_counts())

display(cox_df.head())

Cox dataset: (2836, 7)

Event distribution:
event
1    1760
0    1076
Name: count, dtype: int64


,project_id,start_date,event_date,last_date,event,end_date,duration_months
0,120100067,2015-04-01,2015-04-01,2015-09-01,1,2015-04-01,0.000000
1,160100231,2015-04-01,2015-04-01,2015-07-01,1,2015-04-01,0.000000
2,180100078,2019-06-01,NaT,2022-03-01,0,2022-03-01,32.982917
3,180100210,2015-04-01,2015-04-01,2022-05-01,1,2015-04-01,0.000000
4,180100211,2015-04-01,2015-04-01,2015-05-01,1,2015-04-01,0.000000


In [17]:
# ==========================================
# SURVIVAL DATA QUALITY CHECK
# ==========================================

print("EVENT RATE:",
      round(cox_df["event"].mean(), 4))

print(
    "Censored:",
    (cox_df["event"] == 0).sum()
)

print(
    "Events:",
    (cox_df["event"] == 1).sum()
)

print("\nDuration statistics:")
display(
    cox_df["duration_months"]
    .describe()
    .round(2)
)

print(
    "\nZero/negative durations:",
    (cox_df["duration_months"] <= 0).sum()
)

print(
    "\nMissing durations:",
    cox_df["duration_months"].isna().sum()
)

EVENT RATE: 0.6206
Censored: 1076
Events: 1760

Duration statistics:


count    2836.00
mean       23.97
std        23.76
min         0.00
25%         2.00
50%        17.02
75%        38.99
max       109.00
Name: duration_months, dtype: float64


Zero/negative durations: 615

Missing durations: 0


In [18]:
# ==========================================
# REMOVE PRE-EXISTING DELAYS
# ==========================================

# Projects whose first observation is already delayed
baseline_delay_projects = cox_df.loc[
    cox_df["duration_months"] <= 0,
    "project_id"
]

print("Projects with delay at first observation:",
      len(baseline_delay_projects))

# Keep only projects where the first observed delay
# occurs after the monitoring period begins
cox_incident = cox_df[
    ~cox_df["project_id"].isin(baseline_delay_projects)
].copy()

print("\nOriginal projects:", len(cox_df))
print("Cox projects after exclusion:", len(cox_incident))

Projects with delay at first observation: 615

Original projects: 2836
Cox projects after exclusion: 2221


In [19]:
# ==========================================
# INCIDENT-DELAY SURVIVAL DATA
# ==========================================

print("EVENT DISTRIBUTION:")
print(cox_incident["event"].value_counts())

print("\nEVENT RATE:",
      round(cox_incident["event"].mean(), 4))

print("\nDuration statistics:")
display(
    cox_incident["duration_months"]
    .describe()
    .round(2)
)

print(
    "\nZero/negative durations:",
    (cox_incident["duration_months"] <= 0).sum()
)

print(
    "Missing durations:",
    cox_incident["duration_months"].isna().sum()
)

EVENT DISTRIBUTION:
event
1    1177
0    1044
Name: count, dtype: int64

EVENT RATE: 0.5299

Duration statistics:


count    2221.00
mean       30.61
std        22.75
min         0.92
25%        11.01
50%        27.92
75%        42.97
max       109.00
Name: duration_months, dtype: float64


Zero/negative durations: 0
Missing durations: 0


In [20]:
# ==========================================
# COX DATA SANITY CHECK
# ==========================================

print("=" * 60)
print("COX SURVIVAL DATASET — SANITY CHECK")
print("=" * 60)

print("Projects:", len(cox_incident))
print("Events:", cox_incident["event"].sum())
print("Censored:", (cox_incident["event"] == 0).sum())
print("Event rate:", round(cox_incident["event"].mean(), 4))
print("Minimum duration:",
      round(cox_incident["duration_months"].min(), 2))
print("Maximum duration:",
      round(cox_incident["duration_months"].max(), 2))

assert (cox_incident["duration_months"] > 0).all()

print("\n✅ No zero/negative durations")
print("✅ Incident-delay cohort ready for Cox model")

COX SURVIVAL DATASET — SANITY CHECK
Projects: 2221
Events: 1177
Censored: 1044
Event rate: 0.5299
Minimum duration: 0.92
Maximum duration: 109.0

✅ No zero/negative durations
✅ Incident-delay cohort ready for Cox model


In [21]:
# ==========================================
# PHASE 5.3 — COX BASELINE FEATURES
# ==========================================

cox_features = [
    "project_id",
    "report_month",
    "project_age",
    "original_duration_months",
    "original_cost_crore",
    "current_cost",
    "cumulative_expenditure_crore",
    "milestone_completion_percentage",
    "anticipated_delay_from_original",
    "remaining_org_months",
    "sector_overrun_rate",
    "agency_overrun_rate"
]

# First observed row for each project
baseline_features = (
    df.sort_values("report_month")
      .groupby("project_id")
      .first()
      .reset_index()
)

baseline_features = baseline_features[cox_features].copy()

# Merge baseline features with survival outcome
cox_model_df = cox_incident[
    ["project_id", "duration_months", "event"]
].merge(
    baseline_features,
    on="project_id",
    how="inner"
)

print("Cox model dataset:", cox_model_df.shape)
display(cox_model_df.head())

Cox model dataset: (2221, 14)


,project_id,duration_months,event,report_month,project_age,original_duration_months,original_cost_crore,current_cost,cumulative_expenditure_crore,milestone_completion_percentage,anticipated_delay_from_original,remaining_org_months,sector_overrun_rate,agency_overrun_rate
0,180100078,32.982917,0,2019-06-01,234.0,78.0,578.62,578.62,0.00,NaN,NaN,-156.0,0.363757,0.198413
1,180100221,57.030223,1,2015-04-01,139.0,84.0,6285.33,6285.33,7687.18,NaN,155.0,-55.0,0.238095,0.238095
2,180100261,11.990802,0,2016-08-01,128.0,36.0,358.42,358.42,NaN,0.000000,NaN,-92.0,0.330228,0.308802
3,20100044,2.003942,1,2015-04-01,139.0,84.0,3492.00,5677.00,4967.77,13.636364,72.0,-55.0,0.238095,0.238095
4,220100062,17.049934,0,2015-04-01,300.0,NaN,155.66,155.66,201.26,NaN,NaN,NaN,0.238095,0.238095


In [22]:
# ==========================================
# COX MODEL — CLEAN FEATURES
# ==========================================

cox_model_df = cox_model_df.drop(
    columns=["project_id", "report_month"]
)

# Convert everything to numeric
for col in cox_model_df.columns:
    cox_model_df[col] = pd.to_numeric(
        cox_model_df[col],
        errors="coerce"
    )

# Median imputation for baseline covariates
cox_model_df = cox_model_df.fillna(
    cox_model_df.median(numeric_only=True)
)

print("Missing values:",
      cox_model_df.isna().sum().sum())

print("Final Cox shape:",
      cox_model_df.shape)

display(cox_model_df.describe().round(2))

Missing values: 0
Final Cox shape: (2221, 12)


,duration_months,event,project_age,original_duration_months,original_cost_crore,current_cost,cumulative_expenditure_crore,milestone_completion_percentage,anticipated_delay_from_original,remaining_org_months,sector_overrun_rate,agency_overrun_rate
count,2221.00,2221.00,2221.00,2221.00,2221.00,2221.00,2221.00,2221.00,2221.00,2221.00,2221.00,2221.00
mean,30.61,0.53,32.13,45.30,1129.51,1150.48,127.57,6.60,3.17,15.24,0.21,0.21
std,22.75,0.50,54.75,43.24,3864.00,3994.50,645.93,19.71,22.50,31.51,0.10,0.06
min,0.92,0.00,0.00,1.00,36.80,36.38,0.00,0.00,-36.00,-231.00,0.04,0.07
25%,11.01,0.00,4.00,27.00,255.19,256.66,0.00,0.00,0.00,7.00,0.12,0.20
50%,27.92,1.00,13.00,35.00,416.99,416.99,6.11,0.00,0.00,16.00,0.24,0.24
75%,42.97,1.00,32.00,47.00,858.11,861.06,72.59,0.00,0.00,27.00,0.24,0.24
max,109.00,1.00,409.00,431.00,108000.00,108000.00,22485.00,100.00,374.00,131.00,0.60,0.46


In [26]:
# ==========================================
# COX MODEL — CLEAN SUMMARY
# ==========================================

print("=" * 60)
print("COX PROPORTIONAL HAZARDS MODEL")
print("=" * 60)

print("Observations:", cox_model._n_examples)
print("Events:", cox_model.event_observed.sum())
print("Concordance Index:",
      round(cox_model.concordance_index_, 4))

print("\nCOX COEFFICIENTS")
display(
    cox_model.summary[
        ["coef", "exp(coef)", "se(coef)", "z", "p"]
    ].round(4)
)

COX PROPORTIONAL HAZARDS MODEL
Observations: 2221
Events: 1177
Concordance Index: 0.5861

COX COEFFICIENTS


,coef,exp(coef),se(coef),z,p
covariate,,,,,
project_age,-0.0077,0.9924,0.0010,-7.6395,0.0000
original_duration_months,0.0033,1.0033,0.0011,2.8485,0.0044
original_cost_crore,0.0002,1.0002,0.0001,2.1456,0.0319
current_cost,-0.0002,0.9998,0.0001,-2.2313,0.0257
cumulative_expenditure_crore,0.0002,1.0002,0.0001,4.2962,0.0000
milestone_completion_percentage,0.0042,1.0043,0.0014,3.0075,0.0026
anticipated_delay_from_original,-0.0000,1.0000,0.0011,-0.0262,0.9791
remaining_org_months,-0.0067,0.9933,0.0013,-5.3881,0.0000
sector_overrun_rate,0.6412,1.8987,0.3948,1.6240,0.1044


In [27]:
# ==========================================
# COX MODEL — HAZARD RATIOS
# ==========================================

hazard_ratios = (
    cox_model.hazard_ratios_
    .sort_values(ascending=False)
    .rename("Hazard Ratio")
    .to_frame()
)

print("=" * 60)
print("COX HAZARD RATIOS")
print("=" * 60)

display(hazard_ratios.round(4))

COX HAZARD RATIOS


,Hazard Ratio
covariate,
agency_overrun_rate,56.9729
sector_overrun_rate,1.8987
milestone_completion_percentage,1.0043
original_duration_months,1.0033
cumulative_expenditure_crore,1.0002
original_cost_crore,1.0002
anticipated_delay_from_original,1.0000
current_cost,0.9998
remaining_org_months,0.9933


In [29]:
# ==========================================
# COX MODEL — PH ASSUMPTION CHECK
# ==========================================

print("=" * 60)
print("PROPORTIONAL HAZARDS ASSUMPTION CHECK")
print("=" * 60)

cox_model.check_assumptions(
    cox_model_df,
    p_value_threshold=0.05,
    show_plots=False
)

PROPORTIONAL HAZARDS ASSUMPTION CHECK
The ``p_value_threshold`` is set at 0.05. Even under the null hypothesis of no violations, some
covariates will be below the threshold by chance. This is compounded when there are many covariates.
Similarly, when there are lots of observations, even minor deviances from the proportional hazard
assumption will be flagged.

With that in mind, it's best to use a combination of statistical tests and visual tests to determine
the most serious violations. Produce visual plots using ``check_assumptions(..., show_plots=True)``
and looking for non-constant lines. See link [A] below for a full example.



<lifelines.StatisticalResult: proportional_hazard_test>
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 2221 total observations, 1044 right-censored observations>
         test_name = proportional_hazard_test

---
                                      test_statistic      p  -log2(p)
agency_overrun_rate             km             63.50 <0.005     49.15
                                rank           58.26 <0.005     45.31
anticipated_delay_from_original km             11.61 <0.005     10.57
                                rank           11.58 <0.005     10.55
cumulative_expenditure_crore    km              4.92   0.03      5.23
                                rank            5.27   0.02      5.53
current_cost                    km              0.89   0.35      1.53
                                rank            0.93   0.33      1.58
milestone_completion_percentage km              1.50   0.22      2.18
                                rank            1.87   0.17      2.54
original_cost_crore             km              0.78   0.38      1.41
                                rank            0.79   0.38      1.41
original_duration_months        km             50.84 <0.005     39.86
                                rank           45.86 <0.005     36.20
project_age                     km              7.04   0.01      6.97
                                rank            6.91   0.01      6.87
remaining_org_months            km              0.67   0.41      1.27
                                rank            1.09   0.30      1.76
sector_overrun_rate             km            127.57 <0.005     95.86
                                rank          124.53 <0.005     93.64



1. Variable 'project_age' failed the non-proportional test: p-value is 0.0080.

   Advice 1: the functional form of the variable 'project_age' might be incorrect. That is, there
may be non-linear terms missing. The proportional hazard test used is very sensitive to incorrect
functional forms. See documentation in link [D] below on how to specify a functional form.

   Advice 2: try binning the variable 'project_age' using pd.cut, and then specify it in
`strata=['project_age', ...]` in the call in `.fit`. See documentation in link [B] below.

   Advice 3: try adding an interaction term with your time variable. See documentation in link [C]
below.


2. Variable 'original_duration_months' failed the non-proportional test: p-value is <5e-05.

   Advice 1: the functional form of the variable 'original_duration_months' might be incorrect. That
is, there may be non-linear terms missing. The proportional hazard test used is very sensitive to
incorrect functional forms. See documentation in l

[]

In [32]:
print("survival_df columns:")
print(survival_df.columns.tolist())

print("\nShape:", survival_df.shape)

print("\nFirst 5 rows:")
display(survival_df.head())

survival_df columns:
['project_id', 'report_month', 'anticipated_delay_from_original', 'delay_event']

Shape: (109787, 4)

First 5 rows:


,project_id,report_month,anticipated_delay_from_original,delay_event
0,120100067,2015-04-01,67.0,1
1,120100067,2015-05-01,66.0,1
2,120100067,2015-06-01,66.0,1
3,120100067,2015-07-01,66.0,1
4,120100067,2015-08-01,66.0,1


In [34]:
# ==========================================
# CREATE SURVIVAL DURATION
# ==========================================

survival_df = survival_df.sort_values(
    ["project_id", "report_month"]
).copy()

# Duration = number of months observed from each project's
# first available report to its current report
survival_df["duration_months"] = (
    survival_df.groupby("project_id")["report_month"]
    .transform(lambda x: (x - x.min()).dt.days / 30.44)
)

print("Duration created.")
print(survival_df[[
    "project_id",
    "report_month",
    "duration_months",
    "delay_event"
]].head(10))

print("\nDuration statistics:")
print(survival_df["duration_months"].describe())

Duration created.
  project_id report_month  duration_months  delay_event
0  120100067   2015-04-01         0.000000            1
1  120100067   2015-05-01         0.985545            1
2  120100067   2015-06-01         2.003942            1
3  120100067   2015-07-01         2.989488            1
4  120100067   2015-08-01         4.007884            1
5  120100067   2015-09-01         5.026281            1
6  160100231   2015-04-01         0.000000            1
7  160100231   2015-05-01         0.985545            1
8  160100231   2015-06-01         2.003942            1
9  160100231   2015-07-01         2.989488            1

Duration statistics:
count    109787.000000
mean         27.656731
std          21.083549
min           0.000000
25%          10.019711
50%          23.028909
75%          41.951380
max         109.001314
Name: duration_months, dtype: float64


In [35]:
print("Total rows:", len(survival_df))
print("Unique projects:", survival_df["project_id"].nunique())

print("\nEvent distribution:")
print(survival_df["delay_event"].value_counts(dropna=False))

print("\nProjects with multiple event rows:")
event_counts = survival_df.groupby("project_id")["delay_event"].sum()

print("Projects with >1 event:", (event_counts > 1).sum())
print("Projects with exactly 1 event:", (event_counts == 1).sum())
print("Projects with 0 events:", (event_counts == 0).sum())

print("\nExample project:")
example_id = survival_df.loc[survival_df["delay_event"] == 1, "project_id"].iloc[0]
display(
    survival_df[survival_df["project_id"] == example_id]
    [["project_id", "report_month", "anticipated_delay_from_original", "delay_event"]]
    .head(15)
)

Total rows: 109787
Unique projects: 2836

Event distribution:
delay_event
0    72907
1    36880
Name: count, dtype: int64

Projects with multiple event rows:
Projects with >1 event: 1689
Projects with exactly 1 event: 71
Projects with 0 events: 1076

Example project:


,project_id,report_month,anticipated_delay_from_original,delay_event
0,120100067,2015-04-01,67.0,1
1,120100067,2015-05-01,66.0,1
2,120100067,2015-06-01,66.0,1
3,120100067,2015-07-01,66.0,1
4,120100067,2015-08-01,66.0,1
5,120100067,2015-09-01,66.0,1


In [36]:
# ==========================================
# PHASE 5.3 — PROJECT-LEVEL SURVIVAL DATA
# Time to FIRST delay event
# ==========================================

survival_df = survival_df.sort_values(
    ["project_id", "report_month"]
).copy()

survival_rows = []

for project_id, group in survival_df.groupby("project_id"):

    group = group.sort_values("report_month")

    start_date = group["report_month"].iloc[0]

    # First occurrence of a delay event
    event_rows = group[group["delay_event"] == 1]

    if len(event_rows) > 0:
        # First delay event
        event_date = event_rows["report_month"].iloc[0]

        duration = (event_date - start_date).days / 30.44
        event = 1

    else:
        # No delay observed → right censored
        end_date = group["report_month"].iloc[-1]

        duration = (end_date - start_date).days / 30.44
        event = 0

    survival_rows.append({
        "project_id": project_id,
        "duration_months": duration,
        "event": event
    })

cox_df = pd.DataFrame(survival_rows)

print("PROJECT-LEVEL COX DATA")
print("=" * 50)

print("Observations:", len(cox_df))
print("Events:", cox_df["event"].sum())
print("Censored:", (cox_df["event"] == 0).sum())

print("\nDuration statistics:")
print(cox_df["duration_months"].describe())

print("\nFirst 10 rows:")
display(cox_df.head(10))

PROJECT-LEVEL COX DATA
Observations: 2836
Events: 1760
Censored: 1076

Duration statistics:
count    2836.000000
mean       23.972939
std        23.755113
min         0.000000
25%         2.003942
50%        17.017083
75%        38.994744
max       109.001314
Name: duration_months, dtype: float64

First 10 rows:


,project_id,duration_months,event
0,120100067,0.000000,1
1,160100231,0.000000,1
2,180100078,32.982917,0
3,180100210,0.000000,1
4,180100211,0.000000,1
5,180100221,57.030223,1
6,180100239,0.000000,1
7,180100242,0.000000,1
8,180100243,0.000000,1
9,180100246,0.000000,1


In [37]:
# ==========================================
# PHASE 5.3 — FINAL COX DATASET
# Remove left-truncated zero-duration cases
# and attach baseline project covariates
# ==========================================

# Keep only projects where a positive observation time is available
cox_valid = cox_df[cox_df["duration_months"] > 0].copy()

# Select baseline (first observed) project information
baseline_cols = [
    "project_id",
    "project_age",
    "original_duration_months",
    "original_cost_crore",
    "current_cost",
    "cumulative_expenditure_crore",
    "milestone_completion_percentage",
    "anticipated_delay_from_original",
    "remaining_org_months",
    "sector_overrun_rate",
    "agency_overrun_rate"
]

baseline_df = (
    df.sort_values(["project_id", "report_month"])
      .groupby("project_id", as_index=False)
      .first()[baseline_cols]
)

# Merge baseline covariates with survival outcome
cox_final_df = cox_valid.merge(
    baseline_df,
    on="project_id",
    how="left"
)

print("=" * 55)
print("FINAL COX DATASET")
print("=" * 55)

print("Observations:", len(cox_final_df))
print("Events:", cox_final_df["event"].sum())
print("Censored:", (cox_final_df["event"] == 0).sum())

print("\nZero/negative durations:",
      (cox_final_df["duration_months"] <= 0).sum())

print("\nMissing values:")
print(cox_final_df.isna().sum())

print("\nColumns:")
print(cox_final_df.columns.tolist())

display(cox_final_df.head())

FINAL COX DATASET
Observations: 2221
Events: 1177
Censored: 1044

Zero/negative durations: 0

Missing values:
project_id                            0
duration_months                       0
event                                 0
project_age                           0
original_duration_months            443
original_cost_crore                   0
current_cost                          0
cumulative_expenditure_crore         19
milestone_completion_percentage    1169
anticipated_delay_from_original     451
remaining_org_months                408
sector_overrun_rate                   0
agency_overrun_rate                   0
dtype: int64

Columns:
['project_id', 'duration_months', 'event', 'project_age', 'original_duration_months', 'original_cost_crore', 'current_cost', 'cumulative_expenditure_crore', 'milestone_completion_percentage', 'anticipated_delay_from_original', 'remaining_org_months', 'sector_overrun_rate', 'agency_overrun_rate']


,project_id,duration_months,event,project_age,original_duration_months,original_cost_crore,current_cost,cumulative_expenditure_crore,milestone_completion_percentage,anticipated_delay_from_original,remaining_org_months,sector_overrun_rate,agency_overrun_rate
0,180100078,32.982917,0,234.0,78.0,578.62,578.62,0.00,NaN,NaN,-156.0,0.363757,0.198413
1,180100221,57.030223,1,139.0,84.0,6285.33,6285.33,7687.18,NaN,155.0,-55.0,0.238095,0.238095
2,180100261,11.990802,0,128.0,36.0,358.42,358.42,NaN,0.000000,NaN,-92.0,0.330228,0.308802
3,20100044,2.003942,1,139.0,84.0,3492.00,5677.00,4967.77,13.636364,72.0,-55.0,0.238095,0.238095
4,220100062,17.049934,0,300.0,NaN,155.66,155.66,201.26,NaN,NaN,NaN,0.238095,0.238095


In [38]:
# ==========================================
# PHASE 5.3 — PREPARE COX MODEL FEATURES
# ==========================================

cox_model_df = cox_final_df.copy()

# Project ID is only an identifier, not a predictor
cox_model_df = cox_model_df.drop(columns=["project_id"])

# Predictor columns
cox_features = [
    "project_age",
    "original_duration_months",
    "original_cost_crore",
    "current_cost",
    "cumulative_expenditure_crore",
    "milestone_completion_percentage",
    "anticipated_delay_from_original",
    "remaining_org_months",
    "sector_overrun_rate",
    "agency_overrun_rate"
]

# Median imputation for numeric covariates
for col in cox_features:
    cox_model_df[col] = cox_model_df[col].fillna(
        cox_model_df[col].median()
    )

print("=" * 55)
print("COX MODEL INPUT")
print("=" * 55)

print("Observations:", len(cox_model_df))
print("Events:", cox_model_df["event"].sum())
print("Censored:", (cox_model_df["event"] == 0).sum())

print("\nRemaining missing values:")
print(cox_model_df[cox_features].isna().sum().sum())

print("\nFinal columns:")
print(cox_model_df.columns.tolist())

display(cox_model_df.head())

COX MODEL INPUT
Observations: 2221
Events: 1177
Censored: 1044

Remaining missing values:
0

Final columns:
['duration_months', 'event', 'project_age', 'original_duration_months', 'original_cost_crore', 'current_cost', 'cumulative_expenditure_crore', 'milestone_completion_percentage', 'anticipated_delay_from_original', 'remaining_org_months', 'sector_overrun_rate', 'agency_overrun_rate']


,duration_months,event,project_age,original_duration_months,original_cost_crore,current_cost,cumulative_expenditure_crore,milestone_completion_percentage,anticipated_delay_from_original,remaining_org_months,sector_overrun_rate,agency_overrun_rate
0,32.982917,0,234.0,78.0,578.62,578.62,0.00,0.000000,0.0,-156.0,0.363757,0.198413
1,57.030223,1,139.0,84.0,6285.33,6285.33,7687.18,0.000000,155.0,-55.0,0.238095,0.238095
2,11.990802,0,128.0,36.0,358.42,358.42,6.11,0.000000,0.0,-92.0,0.330228,0.308802
3,2.003942,1,139.0,84.0,3492.00,5677.00,4967.77,13.636364,72.0,-55.0,0.238095,0.238095
4,17.049934,0,300.0,35.0,155.66,155.66,201.26,0.000000,0.0,16.0,0.238095,0.238095


In [41]:
# ==========================================
# PHASE 5.3 — FIT FINAL COX MODEL
# ==========================================

from lifelines import CoxPHFitter

cph = CoxPHFitter()

cph.fit(
    cox_model_df,
    duration_col="duration_months",
    event_col="event"
)

print("=" * 60)
print("FINAL COX PROPORTIONAL HAZARDS MODEL")
print("=" * 60)

print(f"Observations : {cph._n_examples}")
print(f"Events       : {int(cph.event_observed.sum())}")
print(f"Concordance  : {cph.concordance_index_:.4f}")

print("\nCOX MODEL SUMMARY")
display(cph.summary)

FINAL COX PROPORTIONAL HAZARDS MODEL
Observations : 2221
Events       : 1177
Concordance  : 0.5861

COX MODEL SUMMARY


,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
project_age,-0.007657,0.992372,0.001002,-0.009622,-0.005693,0.990424,0.994324,0.0,-7.639450,2.181508e-14,45.381667
original_duration_months,0.003257,1.003262,0.001143,0.001016,0.005498,1.001016,1.005513,0.0,2.848550,4.391899e-03,7.830939
original_cost_crore,0.000188,1.000188,0.000088,0.000016,0.000360,1.000016,1.000360,0.0,2.145583,3.190629e-02,4.970015
current_cost,-0.000196,0.999804,0.000088,-0.000368,-0.000024,0.999632,0.999976,0.0,-2.231341,2.565858e-02,5.284415
cumulative_expenditure_crore,0.000223,1.000223,0.000052,0.000121,0.000325,1.000121,1.000325,0.0,4.296231,1.737262e-05,15.812825
milestone_completion_percentage,0.004250,1.004259,0.001413,0.001480,0.007019,1.001481,1.007044,0.0,3.007456,2.634445e-03,8.568285
anticipated_delay_from_original,-0.000029,0.999971,0.001115,-0.002215,0.002156,0.997788,1.002159,0.0,-0.026153,9.791352e-01,0.030420
remaining_org_months,-0.006746,0.993277,0.001252,-0.009200,-0.004292,0.990842,0.995717,0.0,-5.388145,7.118845e-08,23.743782
sector_overrun_rate,0.641182,1.898723,0.394805,-0.132622,1.414986,0.875796,4.116427,0.0,1.624046,1.043660e-01,3.260276


In [43]:
# Check the variables related to the Cox model
[x for x in globals().keys() if "cox" in x.lower() or "survival" in x.lower()]

['survival_df',
 'cox_df',
 'cox_incident',
 'cox_features',
 'cox_model_df',
 'CoxPHFitter',
 'cox_model',
 'survival_rows',
 'cox_valid',
 'cox_final_df']

In [45]:
# ============================================================
# PHASE 5.4 — COX vs XGBOOST RISK RANKING
# ============================================================

# 1. Cox risk score for each project
cox_risk = cox_model.predict_partial_hazard(
    cox_final_df
).rename("cox_risk")

cox_compare = cox_final_df[["project_id"]].copy()
cox_compare["cox_risk"] = cox_risk.values


# 2. Get latest XGBoost schedule-risk prediction for each project
schedule_compare = df[
    ["project_id", "report_month", "schedule_risk_probability"]
].copy()

schedule_compare = schedule_compare.sort_values(
    ["project_id", "report_month"]
)

schedule_compare = schedule_compare.drop_duplicates(
    subset="project_id",
    keep="last"
)

schedule_compare = schedule_compare.rename(
    columns={"schedule_risk_probability": "xgb_schedule_risk"}
)


# 3. Match the two models
ranking_compare = cox_compare.merge(
    schedule_compare,
    on="project_id",
    how="inner"
)


# 4. Convert both into percentile ranks
ranking_compare["cox_rank"] = ranking_compare["cox_risk"].rank(pct=True)
ranking_compare["xgb_rank"] = ranking_compare["xgb_schedule_risk"].rank(pct=True)


# 5. Rank correlation
spearman_corr = ranking_compare[
    ["cox_rank", "xgb_rank"]
].corr(method="spearman").iloc[0, 1]


print("=" * 60)
print("PHASE 5.4 — COX vs XGBOOST RISK RANKING")
print("=" * 60)

print(f"\nProjects compared      : {len(ranking_compare)}")
print(f"Spearman correlation  : {spearman_corr:.4f}")


# 6. Top 10 according to Cox
print("\nTOP 10 — COX MODEL")

display(
    ranking_compare
    .sort_values("cox_risk", ascending=False)
    [["project_id", "cox_risk", "xgb_schedule_risk"]]
    .head(10)
)


# 7. Top 10 according to XGBoost
print("\nTOP 10 — XGBOOST SCHEDULE MODEL")

display(
    ranking_compare
    .sort_values("xgb_schedule_risk", ascending=False)
    [["project_id", "cox_risk", "xgb_schedule_risk"]]
    .head(10)
)

PHASE 5.4 — COX vs XGBOOST RISK RANKING

Projects compared      : 2221
Spearman correlation  : 0.1280

TOP 10 — COX MODEL


,project_id,cox_risk,xgb_schedule_risk
520,N18000131,8.023482,0.003196
160,N06000189,3.937362,0.001887
271,N16000143,3.712200,0.182935
521,N18000132,3.379003,0.001224
562,N18000192,3.030148,0.392983
523,N18000140,2.768222,0.000277
572,N18000205,2.684207,0.002591
1048,N24000125,2.673991,0.036003
615,N18000252,2.506316,0.808808
163,N06000193,2.442116,0.001575



TOP 10 — XGBOOST SCHEDULE MODEL


,project_id,cox_risk,xgb_schedule_risk
536,N18000163,1.186492,0.997252
542,N18000169,1.176004,0.988681
534,N18000161,1.747170,0.984683
405,N16000325,1.075474,0.982701
613,N18000248,NaN,0.977207
83,N06000102,1.203862,0.977055
755,N22000156,NaN,0.975483
300,N16000203,1.182086,0.975061
532,N18000159,1.454228,0.973965
596,N18000231,NaN,0.973662


In [46]:
# ============================================================
# PHASE 5.4 — DISAGREEMENT ANALYSIS
# ============================================================

# Remove rows where Cox risk could not be calculated
comparison_clean = ranking_compare.dropna(
    subset=["cox_risk", "xgb_schedule_risk"]
).copy()

print("=" * 60)
print("PHASE 5.4 — MODEL DISAGREEMENT ANALYSIS")
print("=" * 60)

print(f"\nValid projects compared : {len(comparison_clean)}")
print(f"Spearman correlation   : {spearman_corr:.4f}")


# ------------------------------------------------------------
# Top-k overlap
# ------------------------------------------------------------

for k in [10, 25, 50, 100]:

    cox_top = set(
        comparison_clean
        .nlargest(k, "cox_risk")["project_id"]
    )

    xgb_top = set(
        comparison_clean
        .nlargest(k, "xgb_schedule_risk")["project_id"]
    )

    overlap = len(cox_top & xgb_top)

    print(
        f"Top-{k} overlap: "
        f"{overlap}/{k} "
        f"({overlap/k:.1%})"
    )


# ------------------------------------------------------------
# Biggest disagreements
# ------------------------------------------------------------

comparison_clean["rank_difference"] = (
    comparison_clean["cox_rank"]
    - comparison_clean["xgb_rank"]
).abs()

print("\nBIGGEST MODEL DISAGREEMENTS")

display(
    comparison_clean
    .nlargest(10, "rank_difference")
    [["project_id",
      "cox_risk",
      "xgb_schedule_risk",
      "cox_rank",
      "xgb_rank",
      "rank_difference"]]
)

PHASE 5.4 — MODEL DISAGREEMENT ANALYSIS

Valid projects compared : 925
Spearman correlation   : 0.1280
Top-10 overlap: 0/10 (0.0%)
Top-25 overlap: 1/25 (4.0%)
Top-50 overlap: 4/50 (8.0%)
Top-100 overlap: 17/100 (17.0%)

BIGGEST MODEL DISAGREEMENTS


,project_id,cox_risk,xgb_schedule_risk,cox_rank,xgb_rank,rank_difference
523,N18000140,2.768222,0.000277,0.994595,0.016659,0.977935
521,N18000132,3.379003,0.001224,0.996757,0.070689,0.926068
163,N06000193,2.442116,0.001575,0.990270,0.082846,0.907425
160,N06000189,3.937362,0.001887,0.998919,0.092301,0.906618
561,N18000190,2.409430,0.002184,0.988108,0.102656,0.885452
572,N18000205,2.684207,0.002591,0.993514,0.109410,0.884103
520,N18000131,8.023482,0.003196,1.000000,0.116614,0.883386
1088,N24000251,1.435939,0.000118,0.881081,0.006754,0.874327
1675,N24000955,1.470668,0.000387,0.896216,0.022512,0.873704
1011,N22000460,1.544197,0.000851,0.921081,0.053129,0.867952


Cox–GBT Risk Ranking Comparison: The Cox survival model showed weak agreement with the XGBoost schedule-risk classifier, with a Spearman rank correlation of 0.128. Top-risk overlap was 0% for the top 10, 4% for the top 25, 8% for the top 50, and 17% for the top 100 projects. This indicates that the two models identify substantially different risk profiles, reflecting their different objectives: XGBoost estimates near-term schedule-risk probability, whereas Cox estimates relative time-to-event hazard.

In [48]:
# ==========================================
# SAVE FINAL FEATURE LIST
# ==========================================

# Our final model uses the 47 columns in X
feature_columns = list(X.columns)

print("Number of features:", len(feature_columns))

joblib.dump(feature_columns, "../models/feature_columns.joblib")

print("✅ Feature list saved")

Number of features: 47
✅ Feature list saved


In [49]:
# ==========================================
# VERIFY PACKAGED FILES
# ==========================================

import os

files_to_check = [
    "../models/cost_final_model.joblib",
    "../models/schedule_final_model.joblib",
    "../models/cox_model.joblib",
    "../models/feature_columns.joblib"
]

for file in files_to_check:
    if os.path.exists(file):
        size_mb = os.path.getsize(file) / (1024 * 1024)
        print(f"✅ {file} — {size_mb:.2f} MB")
    else:
        print(f"❌ {file} — NOT FOUND")

✅ cost_final_model.joblib — 4.25 MB
✅ schedule_final_model.joblib — 4.70 MB
✅ cox_model.joblib — 0.27 MB
✅ feature_columns.joblib — 0.00 MB


In [50]:

import joblib

joblib.dump(cost_final_model, "../models/cost_final_model.joblib")
joblib.dump(schedule_final_model, "../models/schedule_final_model.joblib")
joblib.dump(cox_model, "../models/cox_model.joblib")

joblib.dump(feature_columns, "../models/feature_columns.joblib")

print("✅ Cost model saved")
print("✅ Schedule model saved")
print("✅ Cox model saved")
print("✅ Feature list saved")

✅ Cost model saved
✅ Schedule model saved
✅ Cox model saved
✅ Feature list saved
